# Confidence Intervals

In this section we will be exploring confidence intervals. We will start by simulating many confidence intervals from the standard normal distribution and plotting them alongside the true mean. Confidence intervals are calculated using the following formula:

$$
\bar{x} \pm t \times (\frac{s}{\sqrt n})
$$

As we can observe from this formula, confidence intervals are dependent on the sample size ($n$), standard deviation, expectation, and confidence level. Feel free to play around with these values in the code below.

```{note}
In practice we only use a single confidence interval, but we can visualise the meaning of a confidence interval by simulating many at once. In the code example below, confidence intervals that do not capture the true parameter are coloured red.
```

In [ ]:
import micropip

await micropip.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import ipywidgets as widgets
from ipywidgets import interactive, fixed

In [ ]:
x_values = np.arange(1, 100)
point_estimates = []
ci_lower_bounds = []
ci_upper_bounds = []
n = 1000
standard_deviation = 1
expectation = 3
confidence_level = 0.95

for x in x_values:
    sample = np.random.normal(loc=expectation, scale=standard_deviation, size=n)

    point_estimate = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    t = stats.t.ppf((1 + confidence_level) / 2, n - 1)
    margin_of_error = t * (sample_std / np.sqrt(n))
    ci_lower_bound = point_estimate - margin_of_error
    ci_upper_bound = point_estimate + margin_of_error
    point_estimates.append(point_estimate)
    ci_lower_bounds.append(ci_lower_bound)
    ci_upper_bounds.append(ci_upper_bound)

point_estimates = np.array(point_estimates)
ci_lower_bounds = np.array(ci_lower_bounds)
ci_upper_bounds = np.array(ci_upper_bounds)

lower_errors = point_estimates - ci_lower_bounds
upper_errors = ci_upper_bounds - point_estimates

bounds = np.array([lower_errors, upper_errors])

plt.figure(figsize=(10, 18))

for i, (x_val, point_est, lower_err, upper_err) in enumerate(
    zip(x_values, point_estimates, lower_errors, upper_errors)
):
    color = "green"
    if point_est - lower_err > expectation or point_est + upper_err < expectation:
        color = "red"
    plt.errorbar(
        point_est,
        x_val,
        xerr=[[lower_err], [upper_err]],
        fmt="o",
        capsize=5,
        color=color,
        alpha=0.8,
    )

# plot intervals
plt.xlabel("Estimate")
plt.title("Confidence Intervals for Normal Distribution")
plt.yticks(x_values, [f"Sample {i}" for i in x_values])
plt.grid(True)
plt.axvline(x=expectation, color="r", linestyle="--", label="True Mean")
plt.tight_layout()
plt.show()

As we can see from the simulated intervals, an $n\%$ confidence interval is an indicator that about $n\%$ of intervals would contain the true population mean. In this case, when we simulate 100 confidence intervals at $95\%$ confidence level, we expect about $5$ of the intervals would not contain the true population mean.

Let's explore the impact of sample size, standard deviation, expectation, and confidence level on our confidence intervals. 

In [ ]:
def build_confidence_intervals(n, standard_deviation, expectation, confidence_level):
    point_estimates = []
    ci_lower_bounds = []
    ci_upper_bounds = []

    sample = np.random.normal(loc=expectation, scale=standard_deviation, size=n)

    point_estimate = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    t = stats.t.ppf((1 + confidence_level) / 2, n - 1)
    margin_of_error = t * (sample_std / np.sqrt(n))
    ci_lower_bound = point_estimate - margin_of_error
    ci_upper_bound = point_estimate + margin_of_error
    point_estimates.append(point_estimate)
    ci_lower_bounds.append(ci_lower_bound)
    ci_upper_bounds.append(ci_upper_bound)

    point_estimates = np.array(point_estimates)
    ci_lower_bounds = np.array(ci_lower_bounds)
    ci_upper_bounds = np.array(ci_upper_bounds)

    lower_errors = point_estimates - ci_lower_bounds
    upper_errors = ci_upper_bounds - point_estimates

    return (np.array([lower_errors, upper_errors]), point_estimates)


def update_plot(
    n=1000,
    standard_deviation=1,
    expectation=3,
    confidence_level=0.95,
    title="",
    x_lims=[2, 4],
):
    x_values = np.arange(1, 2)

    bounds, point_estimates = build_confidence_intervals(
        n, standard_deviation, expectation, confidence_level
    )
    plt.figure(figsize=(10, 2))
    plt.errorbar(
        point_estimates,
        x_values,
        xerr=bounds,
        fmt="o",
        capsize=5,
        label="Confidence Intervals",
    )
    plt.xlabel("Estimate")
    plt.xlim(x_lims[0], x_lims[1])
    plt.title(f"Confidence Intervals{title}")
    plt.grid(True)
    plt.axvline(x=expectation, color="r", linestyle="--", label="True Mean")
    plt.legend()
    plt.show()

The sample size has an inverse relationship with the width of the confidence interval. If the sample size is larger, the interval becomes narrower. If the size is smaller, the interval becomes wider. Experiment with the values in the code below to see this relationship.

In [ ]:
sd = 1
exp = 3
conf_level = 0.95

slider_n = widgets.IntSlider(
    value=500, min=10, max=1000, step=10, description="Sample Size"
)
interactive_plot = interactive(
    update_plot,
    n=slider_n,
    standard_deviation=fixed(sd),
    expectation=fixed(exp),
    confidence_level=fixed(conf_level),
    title=fixed(" with varying sample size"),
    x_lims=fixed([2, 4]),
)
interactive_plot

The standard deviation has direct relationship with the width of the confidence interval. A larger standard deviation leads to a wider interval, whilst a smaller standard deviation leads to a narrower interval. Experiment with the values in the code below to see this relationship.

In [ ]:
n_val = 500
exp = 3
conf_level = 0.95

sd_slider = widgets.FloatSlider(
    value=5,
    min=1,
    max=9,
    step=0.1,
    description="Standard Deviation",
    style={"description_width": "120px"},
)
interactive_plot = interactive(
    update_plot,
    n=fixed(n_val),
    standard_deviation=sd_slider,
    expectation=fixed(exp),
    confidence_level=fixed(conf_level),
    title=fixed(" with varying standard deviation"),
    x_lims=fixed([2, 4]),
)
interactive_plot

The expectation shifts the entire confidence interval without impacting the width. Experiment with the values in the code below to see this relationship.

In [ ]:
n_val = 100
sd = 3
conf_level = 0.95

exp_slider = widgets.FloatSlider(
    value=4, min=0, max=8, step=0.1, description="Expectation"
)
interactive_plot = interactive(
    update_plot,
    n=fixed(n_val),
    standard_deviation=fixed(sd),
    expectation=exp_slider,
    confidence_level=fixed(conf_level),
    title=fixed(" with varying expectation"),
    x_lims=fixed([0, 8]),
)
interactive_plot

The confidence level has a direct relationship on the width of the confidence interval. A higher confidence level corresponds to a larger $t$ value, which means a wider interval. A lower confidence level corresponds to a narrower interval. Experiment with the values in the code below to see this relationship.

In [ ]:
n_val = 100
exp = 3
sd = 3

conf_slider = widgets.FloatSlider(
    value=0.82,
    min=0.65,
    max=0.99,
    step=0.01,
    description="Confidence Level",
    style={"description_width": "110px"},
)
interactive_plot = interactive(
    update_plot,
    n=fixed(n_val),
    standard_deviation=fixed(sd),
    expectation=fixed(exp),
    confidence_level=conf_slider,
    title=fixed(" with varying confidence level"),
    x_lims=fixed([2, 4]),
)
interactive_plot

## Bonus: Naïve approximation vs Wilson Method

The naïve approximation that is taught in class uses the following formula to calculate a confidence interval for a proportion:

$$
\left[ \hat{p}-z_{\frac \alpha 2 }\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}, \hat{p}+z_{\frac \alpha 2 }\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}\right]
$$

There is another method called the "Wilson Method" that can also be used to calculate an approximate confidence interval with large $n$ for a proportion. This method involves solving the following formula for $p$ to obtain upper and lower bounds:

$$p_0 = \frac{\hat{p} + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}} $$

In both of these cases, the confidence level is only approximate, and the Wilson method achieves on average a better approximation. The code below simulates confidence intervals calculated using both methods.

In [ ]:
def naive_confidence_interval(x, n, confidence_level=0.95):
    p_hat = x / n
    alpha = 1 - confidence_level
    z = stats.norm.ppf(1 - alpha / 2)
    margin_of_error = z * np.sqrt((p_hat * (1 - p_hat)) / n)

    ci_lower = p_hat - margin_of_error
    ci_upper = p_hat + margin_of_error

    return p_hat, ci_lower, ci_upper

In [ ]:
def wilson_confidence_interval(x, n, confidence_level=0.95):
    alpha = 1 - confidence_level
    z = stats.norm.ppf(1 - alpha / 2)
    p_hat = x / n
    z_squared = z**2

    denominator = 1 + z_squared / n
    sqrt_term = np.sqrt((p_hat * (1 - p_hat) + z_squared / (4 * n)) / n)

    ci_lower = (p_hat + z_squared / (2 * n) - z * sqrt_term) / denominator
    ci_upper = (p_hat + z_squared / (2 * n) + z * sqrt_term) / denominator

    return p_hat, ci_lower, ci_upper

In [ ]:
# calculate intervals for multiple samples
def build_wilson_naive_intervals(num_samples, n, p_true, confidence_level):
    naive_lower = []
    naive_upper = []
    naive_point = []

    wilson_lower = []
    wilson_upper = []
    wilson_point = []

    for i in range(num_samples):
        x = np.random.binomial(n=n, p=p_true)

        p_hat_naive, ci_lower_naive, ci_upper_naive = naive_confidence_interval(
            x, n, confidence_level
        )

        naive_point.append(p_hat_naive)
        naive_lower.append(ci_lower_naive)
        naive_upper.append(ci_upper_naive)

        p_tilde_wilson, ci_lower_wilson, ci_upper_wilson = wilson_confidence_interval(
            x, n, confidence_level
        )

        wilson_point.append(p_tilde_wilson)
        wilson_lower.append(ci_lower_wilson)
        wilson_upper.append(ci_upper_wilson)

    naive_point = np.array(naive_point)
    naive_lower = np.array(naive_lower)
    naive_upper = np.array(naive_upper)

    wilson_point = np.array(wilson_point)
    wilson_lower = np.array(wilson_lower)
    wilson_upper = np.array(wilson_upper)
    return (
        naive_point,
        naive_lower,
        naive_upper,
        wilson_point,
        wilson_lower,
        wilson_upper,
    )

In [ ]:
# plot results
num_samples = 100
n = 1000
p_true = 0.10
confidence_level = 0.95

naive_point, naive_lower, naive_upper, wilson_point, wilson_lower, wilson_upper = (
    build_wilson_naive_intervals(num_samples, n, p_true, confidence_level)
)

naive_lower_errors = naive_point - naive_lower
naive_upper_errors = naive_upper - naive_point

wilson_lower_errors = wilson_point - wilson_lower
wilson_upper_errors = wilson_upper - wilson_point

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 15))

sample_indices = np.arange(1, num_samples + 1)
colors = plt.cm.tab20(np.linspace(0, 1, num_samples))

for i in range(num_samples):
    contains_true = naive_lower[i] <= p_true <= naive_upper[i]
    color = "green" if contains_true else "red"
    alpha = 0.7 if contains_true else 0.9

    ax1.errorbar(
        naive_point[i],
        sample_indices[i],
        xerr=[[naive_lower_errors[i]], [naive_upper_errors[i]]],
        fmt="o",
        capsize=3,
        color=color,
        alpha=alpha,
        markersize=4,
    )

ax1.axvline(
    x=p_true,
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"True Proportion ({p_true})",
)
ax1.set_xlabel("Proportion Estimate")
ax1.set_ylabel("Sample Number")
ax1.set_title(f"Naive Method Confidence Intervals CL={confidence_level:.2f}")
ax1.grid(True, alpha=0.3)
ax1.legend()

for i in range(num_samples):
    contains_true = wilson_lower[i] <= p_true <= wilson_upper[i]
    color = "green" if contains_true else "red"
    alpha = 0.7 if contains_true else 0.9

    ax2.errorbar(
        wilson_point[i],
        sample_indices[i],
        xerr=[[wilson_lower_errors[i]], [wilson_upper_errors[i]]],
        fmt="o",
        capsize=3,
        color=color,
        alpha=alpha,
        markersize=4,
    )

ax2.axvline(
    x=p_true,
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"True Proportion ({p_true})",
)
ax2.set_xlabel("Proportion Estimate")
ax2.set_ylabel("Sample Number")
ax2.set_title(f"Wilson Method Confidence Intervals CL={confidence_level:.2f}")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()